In [1]:
import urllib.request
import json
import os
import ssl

def allowSelfSignedHttps(allowed):
    # bypass the server certificate verification on client side
    if allowed and not os.environ.get('PYTHONHTTPSVERIFY', '') and getattr(ssl, '_create_unverified_context', None):
        ssl._create_default_https_context = ssl._create_unverified_context

allowSelfSignedHttps(True) # this line is needed if you use self-signed certificate in your scoring service.


url = 'https://amlws-001-mistral-7b-endpt.eastus.inference.ml.azure.com/score'
# Replace this with the primary/secondary key or AMLToken for the endpoint
api_key = '<replace with api key>'
if not api_key:
    raise Exception("A key should be provided to invoke the endpoint")

# The azureml-model-deployment header will force the request to go to a specific deployment.
# Remove this header to have the request observe the endpoint traffic rules
headers = {'Content-Type':'application/json', 'Authorization':('Bearer '+ api_key), 'azureml-model-deployment': 'mistralai-mistral-7b-instruct-1' }



In [2]:
# Request data goes here
# The example below assumes JSON formatting which may be updated
# depending on the format your endpoint expects.
# check the test tab in the endpoint interface for a link to API input schema
# More information can be found here:
# https://docs.microsoft.com/azure/machine-learning/how-to-deploy-advanced-entry-script
data = {
    "input_data": {
        "input_string": [
            {
                "role": "user",
                "content": "I am going to Paris, what should I see?"
            },
            {
                "role": "assistant",
                "content": "Paris, the capital of France, is known for its stunning architecture, art museums, historical landmarks, and romantic atmosphere. Here are some of the top attractions to see in Paris:\n\n1. The Eiffel Tower: The iconic Eiffel Tower is one of the most recognizable landmarks in the world and offers breathtaking views of the city.\n2. The Louvre Museum: The Louvre is one of the world's largest and most famous museums, housing an impressive collection of art and artifacts, including the Mona Lisa.\n3. Notre-Dame Cathedral: This beautiful cathedral is one of the most famous landmarks in Paris and is known for its Gothic architecture and stunning stained glass windows.\n\nThese are just a few of the many attractions that Paris has to offer. With so much to see and do, it's no wonder that Paris is one of the most popular tourist destinations in the world."
            },
            {
                "role": "user",
                "content": "What is so great about #1?"
            }
        ],
        "parameters": {
            "temperature": 0.6,
            "top_p": 0.9,
            "do_sample": True,
            "max_new_tokens": 200,
            "return_full_text": False
        }
    }
}

body = str.encode(json.dumps(data))


In [3]:
req = urllib.request.Request(url, body, headers)

try:
    response = urllib.request.urlopen(req)

    result = response.read()
    print(result)
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))

    # Print the headers - they include the requert ID and the timestamp, which are useful for debugging the failure
    print(error.info())
    print(error.read().decode("utf8", 'ignore'))

b'{"output":" The Eiffel Tower is a symbol of Paris and France, and is considered an engineering marvel. It was built between 1887 and 1889 as the entrance arch to the 1889 Exposition Universelle (World\'s Fair) and was originally criticized by some of France\'s leading artists and intellectuals for its design. However, it quickly became a beloved symbol of Paris and France, and today it is visited by millions of tourists every year.\\n\\nOne of the things that makes the Eiffel Tower so great is its impressive size and structure. It stands at over 320 meters (1,050 feet) tall, making it one of the tallest structures in Paris. The tower is made of iron and consists of four pillars that support the four corners of a large square platform. The tower is also famous for its three levels, which offer stunning views of Paris and the surrounding area. Visitors can"}'


In [14]:
# Request data goes here
# The example below assumes JSON formatting which may be updated
# depending on the format your endpoint expects.
# More information can be found here:
# https://docs.microsoft.com/azure/machine-learning/how-to-deploy-advanced-entry-script
data = {
    "input_data": {
        "input_string": [
            {
                "role": "user",
                "content": "What are some common failure modes for pumps? Return your response as JSON."
            }
        ],
        "parameters": {
            "temperature": 0.6,
            "top_p": 0.9,
            "do_sample": True,
            "max_new_tokens": 1000,
            "return_full_text": False
        }
    }
}

body = str.encode(json.dumps(data))


In [15]:
req = urllib.request.Request(url, body, headers)

try:
    response = urllib.request.urlopen(req)

    result = response.read()
    print(result)
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))

    # Print the headers - they include the requert ID and the timestamp, which are useful for debugging the failure
    print(error.info())
    print(error.read().decode("utf8", 'ignore'))

b'{"output":" {\\n\\n\\"failure_modes\\": [\\n  {\\n    \\"name\\": \\"Mechanical Seal Failure\\",\\n    \\"description\\": \\"Mechanical seals are used to prevent leakage between the pump chamber and the environment. Failure of the mechanical seal can lead to significant leaks, resulting in downtime and potential environmental damage.\\",\\n    \\"symptoms\\": [\\"Leaking around the seal area\\", \\"Increased vibration\\", \\"Decreased pump performance\\"]\\n  },\\n  {\\n    \\"name\\": \\"Impeller Damage\\",\\n    \\"description\\": \\"Impellers are responsible for moving fluid through the pump. Damage to the impeller can result in reduced pump performance or complete failure.\\",\\n    \\"symptoms\\": [\\"Decreased flow rate\\", \\"Increased vibration\\", \\"Loud noises during operation\\"]\\n  },\\n  {\\n    \\"name\\": \\"Bearing Failure\\",\\n    \\"description\\": \\"Bearings support the rotating parts of the pump and allow for smooth operation. Failure of the bearings can resul

In [17]:
json.loads(json.loads(result)['output'])

{'failure_modes': [{'name': 'Mechanical Seal Failure',
   'description': 'Mechanical seals are used to prevent leakage between the pump chamber and the environment. Failure of the mechanical seal can lead to significant leaks, resulting in downtime and potential environmental damage.',
   'symptoms': ['Leaking around the seal area',
    'Increased vibration',
    'Decreased pump performance']},
  {'name': 'Impeller Damage',
   'description': 'Impellers are responsible for moving fluid through the pump. Damage to the impeller can result in reduced pump performance or complete failure.',
   'symptoms': ['Decreased flow rate',
    'Increased vibration',
    'Loud noises during operation']},
  {'name': 'Bearing Failure',
   'description': 'Bearings support the rotating parts of the pump and allow for smooth operation. Failure of the bearings can result in increased vibration and eventual pump failure.',
   'symptoms': ['Increased vibration',
    'Loud noises during operation',
    'Decreas

In [18]:
[d['name'] for d in json.loads(json.loads(result)['output'])['failure_modes']]

['Mechanical Seal Failure',
 'Impeller Damage',
 'Bearing Failure',
 'Cavitation',
 'Wear and Tear',
 'Foreign Object Damage',
 'Electrical Failure']

In [20]:
# Request data goes here
# The example below assumes JSON formatting which may be updated
# depending on the format your endpoint expects.
# More information can be found here:
# https://docs.microsoft.com/azure/machine-learning/how-to-deploy-advanced-entry-script

candidate_labels = "thrust bearings issue, pump seal leak, broken breaker, pipe leak, check valve broken"

wo_long_text = """P-5270A high vibrations 12.02.2021 19:59:57 UTC R_MSD365 MS Dynamics 365 (R_MSD365) Notification description: Vibration technician found P-5270A vibrating more than usual. Mechanical notified operations and the pump was swapped to P-5270B 02/18/2021 08:50:09 UTC Dwayne Greenlee (USDGRL) After looking at SKF it looks as though we have an issue with the thrust bearings. The pump needs to be pulled and repaired according to the MRS card
"""
wo_long_text = wo_long_text.replace('\r\n',' ')

user_content = f"Read the Work Order below and label it with one of the following labels: {candidate_labels}\n\n## Work Order:\n{wo_long_text}"

print(user_content)



Read the Work Order below and label it with one of the following labels: thrust bearings issue, pump seal leak, broken breaker, pipe leak, check valve broken

## Work Order:
P-5270A high vibrations 12.02.2021 19:59:57 UTC R_MSD365 MS Dynamics 365 (R_MSD365) Notification description: Vibration technician found P-5270A vibrating more than usual. Mechanical notified operations and the pump was swapped to P-5270B 02/18/2021 08:50:09 UTC Dwayne Greenlee (USDGRL) After looking at SKF it looks as though we have an issue with the thrust bearings. The pump needs to be pulled and repaired according to the MRS card



In [21]:
data = {
    "input_data": {
        "input_string": [
            {
                "role": "user",
                "content": user_content
            }
        ],
        "parameters": {
            "temperature": 0.6,
            "top_p": 0.9,
            "do_sample": True,
            "max_new_tokens": 1000,
            "return_full_text": False
        }
    }
}

body = str.encode(json.dumps(data))

In [22]:
req = urllib.request.Request(url, body, headers)

try:
    response = urllib.request.urlopen(req)

    result = response.read()
    print(result)
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))

    # Print the headers - they include the requert ID and the timestamp, which are useful for debugging the failure
    print(error.info())
    print(error.read().decode("utf8", 'ignore'))

b'{"output":" Thrust bearings issue."}'


In [23]:
json.loads(result)['output']

' Thrust bearings issue.'

### Iterate over rows

In [4]:
def construct_user_content_request_body(wo_text, candidate_labels):
    wo_text = wo_text.replace('\r\n',' ')

    user_content = f"Read the Work Order below and label it with one of the following labels: {candidate_labels}\n\n## Work Order:\n{wo_long_text}"

    data = {
        "input_data": {
            "input_string": [
                {
                    "role": "user",
                    "content": user_content
                }
            ],
            "parameters": {
                "temperature": 0.6,
                "top_p": 0.9,
                "do_sample": True,
                "max_new_tokens": 1000,
                "return_full_text": False
            }
        }
    }

    body = str.encode(json.dumps(data))
    
    return body

In [6]:
candidate_labels = "thrust bearings issue, pump seal leak, broken breaker, pipe leak, check valve broken"

import csv
import pandas as pd

results = []

with open('data/sample_data.csv', mode='r') as csv_file:
    csv_reader = csv.reader(csv_file)
    # line_count = 0
    for i, row in enumerate(csv_reader):
        if i == 0:
            print(f'Column names are {", ".join(row)}')
        elif i < 10:   
            print(i)
            wo_desc = row[0]
            wo_long_text = row[1] # long text description
            failure_mode = row[2]
            # construct request
            body = construct_user_content_request_body(wo_long_text, candidate_labels)
            req = urllib.request.Request(url, body, headers)

            # call model API
            try:
                response = urllib.request.urlopen(req, timeout=10)

                result = response.read()
                predicted_fm = json.loads(result)['output']
                # collect results in row
                results.append((wo_desc, failure_mode, predicted_fm))

            except urllib.error.HTTPError as error:
                print("The request failed with status code: " + str(error.code))

                # Print the headers - they include the requert ID and the timestamp, which are useful for debugging the failure
                print(error.info())
                print(error.read().decode("utf8", 'ignore'))
        else:
            pass

df = pd.DataFrame(results, columns=['WO_Desc','failure_mode_actual','failure_mode_predicted'])


Column names are ﻿Short Description, Long Text Description, Failure Mode Category  
1
2
3
4
5
6
7
8
9


In [7]:
df

,WO_Desc,failure_mode_actual,failure_mode_predicted
0,P-5270A high vibrations,thrust bearings issue,Thrust bearings issue.
1,P-6336A MVR W-438 pump seal leak,pump seal leak,Label: pump seal leak
2,P-5470(D) OIL LEAK ON PIPING TO BEARINGS,pipe leak,pipe leak
3,BKR-221D tripping intermittently,broken breaker,broken breaker.
4,T-1003A steam leak detected,pipe leak,pipe leak
5,CHK-5573 not closing properly,check valve broken,check valve issue.
6,V-4022 abnormal noise,thrust bearings issue,Thrust bearings issue.
7,P-2105B seal failure observed,pump seal leak,Pump seal leak.
8,Pipe L-1010 corrosion hole,pipe leak,pipe leak
